# AlphaBreast V1 - Baseline CNN
## 3-Layer CNN + Single Cross-Attention

Baseline implementation with:
- Simple 3-layer CNN encoder
- Single-direction cross-attention (CC attends to MLO)
- Mass data only (~509 pairs)
- Simple train/test split

Results: 51.0% accuracy, 0.614 AUC

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.metrics import recall_score, f1_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/CBIS-DDSM'
CSV_PATH = os.path.join(DATA_ROOT, 'csv')
JPEG_PATH = os.path.join(DATA_ROOT, 'jpeg')

OUTPUT_DIR = '/content/alphabreast_v1_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# V1: Mass data only
mass_train = pd.read_csv(os.path.join(CSV_PATH, 'mass_case_description_train_set.csv'))
mass_test = pd.read_csv(os.path.join(CSV_PATH, 'mass_case_description_test_set.csv'))

print(f"Training cases: {len(mass_train)}")
print(f"Test cases: {len(mass_test)}")

In [ ]:
def find_image_fixed(jpeg_path, relative_path):
    """Find image from CBIS-DDSM folder structure."""
    if not relative_path or pd.isna(relative_path):
        return None
    parts = str(relative_path).strip().split('/')
    for i in [1, 2]:
        if len(parts) > i:
            folder_path = os.path.join(jpeg_path, parts[i])
            if os.path.exists(folder_path):
                for f in os.listdir(folder_path):
                    if f.endswith(('.jpg', '.jpeg', '.png')):
                        return os.path.join(folder_path, f)
    return None


def create_paired_dataset(df):
    """Create paired CC-MLO samples."""
    paired_samples = []
    grouped = df.groupby(['patient_id', 'left or right breast'])
    for (patient_id, laterality), group in grouped:
        cc_rows = group[group['image view'] == 'CC']
        mlo_rows = group[group['image view'] == 'MLO']
        if len(cc_rows) > 0 and len(mlo_rows) > 0:
            cc_row = cc_rows.iloc[0]
            mlo_row = mlo_rows.iloc[0]
            pathology = str(cc_row['pathology']).upper()
            if 'MALIGNANT' in pathology:
                label = 1
            elif 'BENIGN' in pathology:
                label = 0
            else:
                continue
            paired_samples.append({
                'patient_id': patient_id,
                'cc_cropped': cc_row.get('cropped image file path', ''),
                'mlo_cropped': mlo_row.get('cropped image file path', ''),
                'label': label
            })
    return pd.DataFrame(paired_samples)

paired_train = create_paired_dataset(mass_train)
paired_test = create_paired_dataset(mass_test)
print(f"Training pairs: {len(paired_train)}")
print(f"Test pairs: {len(paired_test)}")

In [ ]:
class CBISDDSMDataset(Dataset):
    """Basic paired dataset for V1."""
    def __init__(self, paired_df, jpeg_path, transform=None):
        self.jpeg_path = jpeg_path
        self.transform = transform
        self.valid_samples = []
        for idx in range(len(paired_df)):
            row = paired_df.iloc[idx]
            cc_path = find_image_fixed(jpeg_path, row.get('cc_cropped', ''))
            mlo_path = find_image_fixed(jpeg_path, row.get('mlo_cropped', ''))
            if cc_path and mlo_path:
                self.valid_samples.append({
                    'cc_path': cc_path,
                    'mlo_path': mlo_path,
                    'label': row['label']
                })
        print(f"Valid samples: {len(self.valid_samples)} / {len(paired_df)}")

    def __len__(self):
        return len(self.valid_samples)

    def __getitem__(self, idx):
        sample = self.valid_samples[idx]
        cc_img = Image.open(sample['cc_path']).convert('L')
        mlo_img = Image.open(sample['mlo_path']).convert('L')
        if self.transform:
            cc_img = self.transform(cc_img)
            mlo_img = self.transform(mlo_img)
        return cc_img, mlo_img, torch.tensor(sample['label'], dtype=torch.long)


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_dataset = CBISDDSMDataset(paired_train, JPEG_PATH, train_transform)
test_dataset = CBISDDSMDataset(paired_test, JPEG_PATH, test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [ ]:
class SimpleCNNEncoder(nn.Module):
    """3-layer CNN encoder for V1 baseline."""
    def __init__(self, out_features=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(128, out_features)

    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.fc(x)


class SimpleAttention(nn.Module):
    """Single-direction attention: CC attends to MLO."""
    def __init__(self, embed_dim=128):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads=4, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, cc_feat, mlo_feat):
        cc = cc_feat.unsqueeze(1)
        mlo = mlo_feat.unsqueeze(1)
        attended, _ = self.attn(cc, mlo, mlo)
        return self.norm(cc + attended).squeeze(1)


class AlphaBreastV1(nn.Module):
    """V1: 3-layer CNN + single cross-attention."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.cc_encoder = SimpleCNNEncoder(128)
        self.mlo_encoder = SimpleCNNEncoder(128)
        self.attention = SimpleAttention(128)
        self.classifier = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, cc, mlo):
        cc_feat = self.cc_encoder(cc)
        mlo_feat = self.mlo_encoder(mlo)
        attended = self.attention(cc_feat, mlo_feat)
        combined = torch.cat([attended, mlo_feat], dim=1)
        return self.classifier(combined)


model = AlphaBreastV1().to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Training
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

best_auc = 0
for epoch in range(30):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for cc, mlo, labels in train_loader:
        cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(cc, mlo)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
    scheduler.step()

    # Evaluate
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for cc, mlo, labels in test_loader:
            cc, mlo, labels = cc.to(device), mlo.to(device), labels.to(device)
            outputs = model(cc, mlo)
            probs = F.softmax(outputs, dim=1)
            _, pred = outputs.max(1)
            all_preds.extend(pred.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds) * 100
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.5

    if auc > best_auc:
        best_auc = auc
        best_acc = acc

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/30 | Train Acc: {100.*correct/total:.1f}% | "
              f"Test Acc: {acc:.1f}%, AUC: {auc:.3f}")

print(f"\nV1 Best: Acc={best_acc:.1f}%, AUC={best_auc:.3f}")